In [1]:
import pickle
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

with open('/lp-dev/24BCE1793/data/processed/all_graphs_29sub.pkl', 'rb') as f:
    graphs = pickle.load(f)

# Build feature matrix
# Node features: mean relative band power across electrodes (4 features)
# Edge features: mean PLV per band across all edges (4 features)
X, y = [], []
for g in graphs:
    node_feat = g.x.numpy().mean(axis=0)      # (4,) mean across 62 electrodes
    edge_feat = g.edge_attr.numpy().mean(axis=0)  # (4,) mean PLV per band
    X.append(np.concatenate([node_feat, edge_feat]))
    y.append(g.y.item())

X = np.array(X)
y = np.array(y)
print(f"Feature matrix: {X.shape}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for clf, name in [(LogisticRegression(max_iter=1000), 'LogReg'),
                  (RandomForestClassifier(n_estimators=100, random_state=42), 'RandomForest')]:
    scores = []
    for train_idx, test_idx in cv.split(X_scaled, y):
        clf.fit(X_scaled[train_idx], y[train_idx])
        scores.append(accuracy_score(y[test_idx], clf.predict(X_scaled[test_idx])))
    print(f"{name}: {np.mean(scores)*100:.2f}% ± {np.std(scores)*100:.2f}%")

print(f"Chance: 33.33%")

/lp-dev/24BCE1793/fignn_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature matrix: (25056, 8)
LogReg: 39.26% ± 1.23%
RandomForest: 47.12% ± 0.47%
Chance: 33.33%


In [2]:
# Test PLV-only vs power-only vs combined on new graphs
for feat_name, feat_slice in [('power only', slice(0,4)), 
                                ('PLV only',   slice(4,8)),
                                ('combined',   slice(0,8))]:
    scores = []
    X_sub = X_scaled[:, feat_slice]
    for train_idx, test_idx in cv.split(X_sub, y):
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_sub[train_idx], y[train_idx])
        scores.append(accuracy_score(y[test_idx], rf.predict(X_sub[test_idx])))
    print(f"RF {feat_name}: {np.mean(scores)*100:.2f}%")

RF power only: 42.73%
RF PLV only: 40.65%
RF combined: 47.12%


In [3]:
# Quick test — correlation as edge features
import numpy as np
from scipy.stats import pearsonr

# Recompute edge features with correlation for existing graphs
X_corr = []
for g in graphs:
    node_feat = g.x.numpy().mean(axis=0)          # (4,) relative power
    plv_feat  = g.edge_attr.numpy()[:, :4].mean(axis=0)  # (4,) mean PLV
    # We don't have corr yet so test with PLV+power as proxy
    X_corr.append(np.concatenate([node_feat, plv_feat]))

X_corr   = np.array(X_corr)
X_scaled2 = StandardScaler().fit_transform(X_corr)

scores = []
for train_idx, test_idx in cv.split(X_scaled2, y):
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_scaled2[train_idx], y[train_idx])
    scores.append(accuracy_score(y[test_idx], rf.predict(X_scaled2[test_idx])))
print(f"RF power+PLV sanity check: {np.mean(scores)*100:.2f}%")

RF power+PLV sanity check: 47.12%


In [4]:
import sys
sys.path.insert(0, '/home/nvidia/24BCE1793/fignn_cni__project')
from src.preprocessing.preprocess import process_file
from src.preprocessing.plv import batch_compute_plv
from src.preprocessing.graph_builder import build_graph_dataset
from src.models.model import FIGNNEncoder
from torch_geometric.loader import DataLoader
import torch

result = process_file(
    '/lp-dev/24BCE1793/data/raw/sub-01/ses-S1/eeg/zeroBACK.set', 0)
conn_list = batch_compute_plv(result['band_epochs'], verbose=False)
graphs    = build_graph_dataset(result, conn_list)

print(f"Edge attr shape: {graphs[0].edge_attr.shape}")  # should be (n_edges, 8)

model  = FIGNNEncoder()
loader = DataLoader(graphs[:10], batch_size=10)
batch  = next(iter(loader))
z      = model(batch)
print(f"z shape: {z.shape}")   # should be (10, 64)
print(f"Band weights: {model.get_band_importance()}")

  Loading: zeroBACK.set  label=0


/home/nvidia/24BCE1793/fignn_cni__project/src/preprocessing/preprocess.py:37: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/home/nvidia/24BCE1793/fignn_cni__project/src/preprocessing/preprocess.py:37: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
/home/nvidia/24BCE1793/fignn_cni__project/src/preprocessing/preprocess.py:37: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)


    → 62 ch  |  96 trials (code 6021)  |  500.0 Hz
    Built 96 graphs  |  sample edges: 936  |  label: 0
Edge attr shape: torch.Size([936, 8])
z shape: torch.Size([10, 64])
Band weights: {'plv': {'delta': 0.25, 'theta': 0.25, 'alpha': 0.25, 'beta': 0.25}, 'corr': {'delta': 0.25, 'theta': 0.25, 'alpha': 0.25, 'beta': 0.25}}


In [5]:
import pickle
from collections import Counter
with open('/lp-dev/24BCE1793/data/processed/all_graphs_29sub.pkl', 'rb') as f:
    graphs = pickle.load(f)

print(f"Total: {len(graphs)}")
print(f"Labels: {Counter(g.y.item() for g in graphs)}")
print(f"Node shape: {graphs[0].x.shape}")
print(f"Edge attr shape: {graphs[0].edge_attr.shape}")  # should be (n_edges, 8)
print(f"Edge index shape: {graphs[0].edge_index.shape}")

Total: 25056
Labels: Counter({0: 8352, 1: 8352, 2: 8352})
Node shape: torch.Size([62, 4])
Edge attr shape: torch.Size([936, 8])
Edge index shape: torch.Size([2, 936])


In [19]:
import pickle
import numpy as np
from collections import Counter

PATH = "/lp-dev/24BCE1793/data/processed/all_graphs_29sub.pkl"

with open(PATH, "rb") as f:
    graphs = pickle.load(f)

print("Graphs:", len(graphs))
print("Labels:", Counter(g.y.item() for g in graphs))
print("x:", graphs[0].x.shape)
print("edge_attr:", graphs[0].edge_attr.shape)

Graphs: 25056
Labels: Counter({0: 8352, 1: 8352, 2: 8352})
x: torch.Size([62, 4])
edge_attr: torch.Size([936, 8])


In [20]:
Xp = []
Xplv = []
Xcorr = []
y = []

for g in graphs:

    # Mean relative band power across 62 electrodes
    power = g.x.numpy().mean(axis=0)

    edge = g.edge_attr.numpy()

    # First 4 = PLV
    plv = edge[:, :4].mean(axis=0)

    # Next 4 = correlation/covariance-derived edge features
    corr = edge[:, 4:8].mean(axis=0)

    Xp.append(power)
    Xplv.append(plv)
    Xcorr.append(corr)
    y.append(g.y.item())

Xp    = np.asarray(Xp)
Xplv  = np.asarray(Xplv)
Xcorr = np.asarray(Xcorr)
y     = np.asarray(y)

print("Power:", Xp.shape)
print("PLV:", Xplv.shape)
print("Corr:", Xcorr.shape)
print("Labels:", y.shape)

print("\nFirst sample")
print("Power:", Xp[0])
print("PLV:", Xplv[0])
print("Corr:", Xcorr[0])

Power: (25056, 4)
PLV: (25056, 4)
Corr: (25056, 4)
Labels: (25056,)

First sample
Power: [0.29779375 0.14832057 0.21557845 0.33830723]
PLV: [0.63544506 0.5754894  0.6623132  0.5910582 ]
Corr: [0.23129271 0.2684047  0.2377531  0.23803481]


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

experiments = {
    "Power only": Xp,
    "PLV only": Xplv,
    "Corr only": Xcorr,

    "Power + PLV":
        np.concatenate([Xp, Xplv], axis=1),

    "Power + Corr":
        np.concatenate([Xp, Xcorr], axis=1),

    "PLV + Corr":
        np.concatenate([Xplv, Xcorr], axis=1),

    "Power + PLV + Corr":
        np.concatenate([Xp, Xplv, Xcorr], axis=1),
}

for name, X in experiments.items():

    Xtr, Xte, ytr, yte = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    rf = RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    )

    rf.fit(Xtr, ytr)

    pred = rf.predict(Xte)

    acc = accuracy_score(yte, pred)

    print(f"{name:22s}: {acc*100:.2f}%")

Power only            : 42.68%
PLV only              : 39.49%
Corr only             : 39.68%
Power + PLV           : 46.11%
Power + Corr          : 45.27%
PLV + Corr            : 42.80%
Power + PLV + Corr    : 47.31%


In [9]:
import sys
import torch
import torch.nn as nn

sys.path.insert(
    0,
    "/home/nvidia/24BCE1793/fignn_cni__project"
)

from src.models.model import FIGNNEncoder

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [13]:
class FIGNNClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = FIGNNEncoder()

        self.head = nn.Linear(64, 3)

    def forward(self, data):

        z = self.encoder(data)

        logits = self.head(z)

        return logits, z


model = FIGNNClassifier().to(device)

print(
    "Parameters:",
    sum(p.numel() for p in model.parameters()
        if p.requires_grad)
)

Parameters: 85067


In [14]:
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
import numpy as np

labels = np.array([
    g.y.item() for g in graphs
])

idx = np.arange(len(graphs))

train_idx, val_idx = train_test_split(
    idx,
    test_size=0.15,
    stratify=labels,
    random_state=42
)

train_set = [graphs[i] for i in train_idx]
val_set   = [graphs[i] for i in val_idx]

train_loader = DataLoader(
    train_set,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_set,
    batch_size=128,
    shuffle=False
)

print(len(train_set), len(val_set))

21297 3759


In [16]:
batch = next(iter(train_loader)).to(device)

with torch.no_grad():
    z = model.encoder(batch)

print("Embedding shape:", z.shape)

Embedding shape: torch.Size([128, 64])


In [17]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

criterion = nn.CrossEntropyLoss()

for epoch in range(1, 31):

    # TRAIN
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        logits, z = model(batch)

        target = batch.y.view(-1)

        loss = criterion(logits, target)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        pred = logits.argmax(1)

        train_correct += (
            pred == target
        ).sum().item()

        train_total += target.size(0)


    # VALIDATION
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch in val_loader:

            batch = batch.to(device)

            logits, z = model(batch)

            target = batch.y.view(-1)

            pred = logits.argmax(1)

            val_correct += (
                pred == target
            ).sum().item()

            val_total += target.size(0)


    train_acc = train_correct / train_total
    val_acc   = val_correct / val_total

    print(
        f"Epoch {epoch:02d} | "
        f"loss={train_loss/len(train_loader):.4f} | "
        f"train={train_acc*100:.2f}% | "
        f"val={val_acc*100:.2f}%"
    )

Epoch 01 | loss=1.0810 | train=39.04% | val=39.16%
Epoch 02 | loss=1.0800 | train=39.14% | val=39.21%
Epoch 03 | loss=1.0793 | train=39.22% | val=37.96%
Epoch 04 | loss=1.0790 | train=39.21% | val=38.52%
Epoch 05 | loss=1.0784 | train=39.66% | val=38.81%
Epoch 06 | loss=1.0779 | train=39.65% | val=38.95%
Epoch 07 | loss=1.0779 | train=39.55% | val=38.33%
Epoch 08 | loss=1.0780 | train=39.37% | val=39.35%
Epoch 09 | loss=1.0773 | train=39.86% | val=39.05%
Epoch 10 | loss=1.0762 | train=39.68% | val=39.13%
Epoch 11 | loss=1.0766 | train=39.75% | val=38.39%
Epoch 12 | loss=1.0762 | train=39.87% | val=38.87%
Epoch 13 | loss=1.0758 | train=39.63% | val=39.27%
Epoch 14 | loss=1.0755 | train=40.11% | val=38.68%
Epoch 15 | loss=1.0754 | train=40.05% | val=38.95%
Epoch 16 | loss=1.0756 | train=40.27% | val=38.39%
Epoch 17 | loss=1.0753 | train=40.26% | val=39.11%
Epoch 18 | loss=1.0748 | train=40.03% | val=38.95%
Epoch 19 | loss=1.0745 | train=39.95% | val=39.58%
Epoch 20 | loss=1.0743 | train=

In [18]:
import pickle
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from torch.utils.data import TensorDataset, DataLoader

# --------------------------------------------------
# Load graphs
# --------------------------------------------------

with open(
    "/lp-dev/24BCE1793/data/processed/all_graphs_29sub.pkl",
    "rb"
) as f:
    graphs = pickle.load(f)

# --------------------------------------------------
# Same 12 features used by RF
# --------------------------------------------------

X = []
y = []

for g in graphs:

    power = g.x.mean(dim=0).numpy()

    plv = g.edge_attr[:, :4].mean(dim=0).numpy()

    corr = g.edge_attr[:, 4:8].mean(dim=0).numpy()

    X.append(np.concatenate([power, plv, corr]))
    y.append(g.y.item())

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int64)

print("X:", X.shape)
print("y:", y.shape)

# --------------------------------------------------
# Same random split
# --------------------------------------------------

Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=42
)

# --------------------------------------------------
# Standardise
# --------------------------------------------------

scaler = StandardScaler()

Xtr = scaler.fit_transform(Xtr)
Xte = scaler.transform(Xte)

train_ds = TensorDataset(
    torch.tensor(Xtr, dtype=torch.float32),
    torch.tensor(ytr, dtype=torch.long)
)

test_ds = TensorDataset(
    torch.tensor(Xte, dtype=torch.float32),
    torch.tensor(yte, dtype=torch.long)
)

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=256
)

# --------------------------------------------------
# Tiny MLP
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = nn.Sequential(
    nn.Linear(12, 64),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 3)
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

criterion = nn.CrossEntropyLoss()

# --------------------------------------------------
# Train
# --------------------------------------------------

for epoch in range(1, 31):

    model.train()

    correct = 0
    total = 0
    total_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)

        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

        pred = logits.argmax(dim=1)

        correct += (pred == yb).sum().item()
        total += yb.size(0)

    train_acc = correct / total

    # Validation
    model.eval()

    preds = []
    true = []

    with torch.no_grad():

        for xb, yb in test_loader:

            xb = xb.to(device)

            logits = model(xb)

            preds.extend(
                logits.argmax(dim=1).cpu().numpy()
            )

            true.extend(yb.numpy())

    val_acc = accuracy_score(true, preds)

    print(
        f"Epoch {epoch:02d} | "
        f"loss={total_loss/total:.4f} | "
        f"train={train_acc*100:.2f}% | "
        f"val={val_acc*100:.2f}%"
    )

X: (25056, 12)
y: (25056,)
Epoch 01 | loss=1.0882 | train=37.47% | val=39.21%
Epoch 02 | loss=1.0765 | train=40.09% | val=40.78%
Epoch 03 | loss=1.0729 | train=40.96% | val=40.17%
Epoch 04 | loss=1.0687 | train=40.94% | val=41.26%
Epoch 05 | loss=1.0667 | train=41.80% | val=41.18%
Epoch 06 | loss=1.0656 | train=42.02% | val=42.40%
Epoch 07 | loss=1.0620 | train=42.40% | val=41.77%
Epoch 08 | loss=1.0632 | train=42.07% | val=41.05%
Epoch 09 | loss=1.0608 | train=42.38% | val=43.02%
Epoch 10 | loss=1.0579 | train=42.87% | val=42.43%
Epoch 11 | loss=1.0580 | train=43.09% | val=43.26%
Epoch 12 | loss=1.0556 | train=43.14% | val=43.04%
Epoch 13 | loss=1.0552 | train=43.13% | val=43.42%
Epoch 14 | loss=1.0526 | train=43.82% | val=44.16%
Epoch 15 | loss=1.0517 | train=44.00% | val=43.02%
Epoch 16 | loss=1.0508 | train=43.82% | val=44.05%
Epoch 17 | loss=1.0490 | train=44.40% | val=43.44%
Epoch 18 | loss=1.0505 | train=43.88% | val=43.63%
Epoch 19 | loss=1.0467 | train=44.39% | val=43.39%
Epoc

In [21]:
# Run this in a notebook cell after loading graphs
import torch, pickle
with open('/lp-dev/24BCE1793/data/processed/all_graphs_29sub.pkl', 'rb') as f:
    graphs = pickle.load(f)
all_x = torch.cat([g.x for g in graphs], dim=0)
print(f"Post-norm mean: {all_x.mean(dim=0)}")
print(f"Post-norm std:  {all_x.std(dim=0)}")

Post-norm mean: tensor([0.4820, 0.1825, 0.1706, 0.1649])
Post-norm std:  tensor([0.2117, 0.0785, 0.1210, 0.1256])


In [1]:
import pickle, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

with open('/lp-dev/24BCE1793/data/processed/all_graphs_29sub.pkl', 'rb') as f:
    graphs = pickle.load(f)

X, y = [], []
for g in graphs:
    node_feat = g.x.numpy().mean(axis=0)           # (4,) relative power
    plv_feat  = g.edge_attr.numpy()[:, :4].mean(axis=0)  # (4,) PLV
    corr_feat = g.edge_attr.numpy()[:, 4:].mean(axis=0)  # (4,) correlation
    X.append(np.concatenate([node_feat, plv_feat, corr_feat]))
    y.append(g.y.item())

X = np.array(X)
y = np.array(y)
print(f"Feature matrix: {X.shape}")

scaler = StandardScaler()
X_s    = scaler.fit_transform(X)
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for feat_name, cols in [
    ('power only',       slice(0,4)),
    ('PLV only',         slice(4,8)),
    ('corr only',        slice(8,12)),
    ('power+PLV',        slice(0,8)),
    ('power+corr',       np.r_[0:4, 8:12]),
    ('PLV+corr',         slice(4,12)),
    ('all (power+PLV+corr)', slice(0,12)),
]:
    Xi = X_s[:, cols] if isinstance(cols, slice) else X_s[:, cols]
    scores = []
    for tr, te in cv.split(Xi, y):
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(Xi[tr], y[tr])
        scores.append(accuracy_score(y[te], rf.predict(Xi[te])))
    print(f"RF {feat_name}: {np.mean(scores)*100:.2f}% ± {np.std(scores)*100:.2f}%")

print("Chance: 33.33%")

/lp-dev/24BCE1793/fignn_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature matrix: (25056, 12)
RF power only: 42.73% ± 0.40%
RF PLV only: 40.65% ± 0.42%
RF corr only: 39.34% ± 0.89%
RF power+PLV: 47.12% ± 0.47%
RF power+corr: 45.33% ± 0.61%
RF PLV+corr: 41.93% ± 0.36%
RF all (power+PLV+corr): 47.87% ± 0.37%
Chance: 33.33%
